In [0]:
import pyspark.sql.functions as F

# Reading Hospital A patient data
df_hosa = spark.read.parquet("abfss://bronze@ttadlsjcrh.dfs.core.windows.net/hosa/patients")

# Simulando un cambio de dirección del paciente HOSP1-000001
# df_hosa = df_hosa.withColumn(
#     "Address",
#     F.when(F.col("PatientID") == "HOSP1-000001", "2701 Pablo Kisel Blvd, Brownsville, TX 78526").otherwise(F.col("Address"))
# )

df_hosa.createOrReplaceTempView("patients_hosa")

# Reading Hospital B patient data
df_hosb = spark.read.parquet("abfss://bronze@ttadlsjcrh.dfs.core.windows.net/hosb/patients")
df_hosb.createOrReplaceTempView("patients_hosb")

In [0]:
import pyspark.sql.functions as F

# Reading Hospital A patient data
df_hosa = spark.read.parquet("abfss://bronze@ttadlsjcrh.dfs.core.windows.net/hosa/patients")

df_hosa.createOrReplaceTempView("patients_hosa")

# Reading Hospital B patient data
df_hosb = spark.read.parquet("abfss://bronze@ttadlsjcrh.dfs.core.windows.net/hosb/patients")
df_hosb.createOrReplaceTempView("patients_hosb")

In [0]:
%sql
SELECT PatientID, FirstName, LastName, Address
FROM patients_hosa
LIMIT 5

PatientID,FirstName,LastName,Address


In [0]:
%sql
SELECT * FROM patients_hosb
LIMIT 5

ID,F_Name,L_Name,M_Name,SSN,PhoneNumber,Gender,DOB,Address,Updated_Date,datasource


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW cdm_patients AS
SELECT CONCAT(SRC_PatientID,'-', datasource) AS Patient_Key, *
FROM (
    SELECT
    PatientID AS SRC_PatientID ,
    FirstName,
    LastName,
    MiddleName,
    SSN,
    PhoneNumber,
    Gender,
    DOB,
    Address,
    ModifiedDate,
    datasource
    FROM patients_hosa
    UNION ALL
    SELECT
    ID AS SRC_PatientID,
    F_Name AS FirstName,
    L_Name AS LastName,
    M_Name ASMiddleName,
    SSN,
    PhoneNumber,
    Gender,
    DOB,
    Address,
    Updated_Date AS ModifiedDate,
    datasource
     FROM patients_hosb
)

In [0]:
%sql
SELECT * FROM cdm_patients
LIMIT 5

Patient_Key,SRC_PatientID,FirstName,LastName,MiddleName,SSN,PhoneNumber,Gender,DOB,Address,ModifiedDate,datasource
HOSP1-000001-hos-a,HOSP1-000001,Rick,Russo,U,188-23-9828,+1-630-829-7585x0769,Female,1937-06-04,"Unit 0915 Box 7064, DPO AA 82777",2020-05-25,hos-a
HOSP1-000002-hos-a,HOSP1-000002,Gregory,Graham,B,730-45-8217,456.746.7289x69233,Female,1937-06-10,"9864 Gibson Islands, Danielside, KY 99809",2021-06-05,hos-a
HOSP1-000003-hos-a,HOSP1-000003,Mary,Ryan,H,348-14-7947,522-501-5461,Female,1926-08-09,"6194 Joseph Turnpike, North Juan, OH 46800",2024-09-06,hos-a
HOSP1-000004-hos-a,HOSP1-000004,Daniel,Brown,D,013-38-1645,+1-345-608-9409,Male,1971-10-23,"780 Conrad Isle, Pricebury, KS 61167",2022-04-07,hos-a
HOSP1-000005-hos-a,HOSP1-000005,Brad,Carroll,M,461-53-6290,963.994.2969x6232,Male,1927-10-18,"3167 Hall Burg, Tannertown, IL 03017",2022-06-19,hos-a


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW quality_checks AS
SELECT
    Patient_Key,
    SRC_PatientID,
    FirstName,
    LastName,
    MiddleName,
    SSN,
    PhoneNumber,
    Gender,
    DOB,
    Address,
    ModifiedDate As SRC_ModifiedDate,
    datasource,
    CASE 
        WHEN SRC_PatientID IS NULL OR dob IS NULL OR firstname IS NULL or lower(firstname)='null' THEN TRUE
        ELSE FALSE
    END AS is_quarantined
FROM cdm_patients

In [0]:
%sql
CREATE TABLE IF NOT EXISTS silver.patients (
    Patient_Key STRING,
    SRC_PatientID STRING,
    FirstName STRING,
    LastName STRING,
    MiddleName STRING,
    SSN STRING,
    PhoneNumber STRING,
    Gender STRING,
    DOB DATE,
    Address STRING,
    SRC_ModifiedDate TIMESTAMP,
    datasource STRING,
    is_quarantined BOOLEAN,
    inserted_date TIMESTAMP,
    modified_date TIMESTAMP,
    is_current BOOLEAN
)
USING DELTA;

In [0]:
%sql
SELECT * FROM silver.patients

Patient_Key,SRC_PatientID,FirstName,LastName,MiddleName,SSN,PhoneNumber,Gender,DOB,Address,SRC_ModifiedDate,datasource,is_quarantined,inserted_date,modified_date,is_current
HOSP1-000001-hos-a,HOSP1-000001,Rick,Russo,U,188-23-9828,+1-630-829-7585x0769,Female,1937-06-04,"Unit 0915 Box 7064, DPO AA 82777",2020-05-25T00:00:00.000Z,hos-a,false,2025-12-30T23:48:53.806Z,2025-12-30T23:48:53.806Z,true
HOSP1-000002-hos-a,HOSP1-000002,Gregory,Graham,B,730-45-8217,456.746.7289x69233,Female,1937-06-10,"9864 Gibson Islands, Danielside, KY 99809",2021-06-05T00:00:00.000Z,hos-a,false,2025-12-30T23:48:53.806Z,2025-12-30T23:48:53.806Z,true
HOSP1-000003-hos-a,HOSP1-000003,Mary,Ryan,H,348-14-7947,522-501-5461,Female,1926-08-09,"6194 Joseph Turnpike, North Juan, OH 46800",2024-09-06T00:00:00.000Z,hos-a,false,2025-12-30T23:48:53.806Z,2025-12-30T23:48:53.806Z,true
HOSP1-000004-hos-a,HOSP1-000004,Daniel,Brown,D,013-38-1645,+1-345-608-9409,Male,1971-10-23,"780 Conrad Isle, Pricebury, KS 61167",2022-04-07T00:00:00.000Z,hos-a,false,2025-12-30T23:48:53.806Z,2025-12-30T23:48:53.806Z,true
HOSP1-000005-hos-a,HOSP1-000005,Brad,Carroll,M,461-53-6290,963.994.2969x6232,Male,1927-10-18,"3167 Hall Burg, Tannertown, IL 03017",2022-06-19T00:00:00.000Z,hos-a,false,2025-12-30T23:48:53.806Z,2025-12-30T23:48:53.806Z,true
HOSP1-000006-hos-a,HOSP1-000006,Melissa,Lester,Y,359-88-6883,511-940-3097x95946,Female,1994-03-22,"542 Rogers Divide, South Amy, RI 23000",2022-09-24T00:00:00.000Z,hos-a,false,2025-12-30T23:48:53.806Z,2025-12-30T23:48:53.806Z,true
HOSP1-000007-hos-a,HOSP1-000007,Stephanie,Griffin,Q,879-32-6527,313.753.8992x0867,Male,1957-03-10,"3952 Le Highway, Frenchbury, SC 21179",2021-06-16T00:00:00.000Z,hos-a,false,2025-12-30T23:48:53.806Z,2025-12-30T23:48:53.806Z,true
HOSP1-000008-hos-a,HOSP1-000008,Sandra,Holt,L,148-25-8251,+1-335-229-5080,Female,2024-07-31,"5102 Thompson Locks Suite 182, Port Jenniferborough, NC 93111",2020-03-23T00:00:00.000Z,hos-a,false,2025-12-30T23:48:53.806Z,2025-12-30T23:48:53.806Z,true
HOSP1-000009-hos-a,HOSP1-000009,Austin,Cline,H,219-06-7294,001-899-655-3369x486,Male,2001-11-16,"465 Flores Forest, Port Paul, NJ 44109",2024-05-22T00:00:00.000Z,hos-a,false,2025-12-30T23:48:53.806Z,2025-12-30T23:48:53.806Z,true
HOSP1-000010-hos-a,HOSP1-000010,Ryan,Hall,A,425-22-5742,(962)707-1206x023,Female,1983-05-06,"915 Joseph Walks Suite 592, Rhodesberg, FL 43049",2020-01-07T00:00:00.000Z,hos-a,false,2025-12-30T23:48:53.806Z,2025-12-30T23:48:53.806Z,true


In [0]:
%sql
-- Step 1: Mark existing records as historical (is_current = false) for patients that will be updated
MERGE INTO silver.patients AS target
USING quality_checks AS source
ON target.Patient_Key = source.Patient_Key
AND target.is_current = true
WHEN MATCHED
AND (
    target.SRC_PatientID <> source.SRC_PatientID OR
    target.FirstName <> source.FirstName OR
    target.LastName <> source.LastName OR
    target.MiddleName <> source.MiddleName OR
    target.SSN <> source.SSN OR
    target.PhoneNumber <> source.PhoneNumber OR
    target.Gender <> source.Gender OR
    target.DOB <> source.DOB OR
    target.Address <> source.Address OR
    target.SRC_ModifiedDate <> source.SRC_ModifiedDate OR
    target.datasource <> source.datasource OR
    target.is_quarantined <> source.is_quarantined
)
THEN UPDATE SET
    target.is_current = false,
    target.modified_date = current_timestamp()

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
%sql
-- select * from quality_checks
-- where Patient_Key = 'HOSP1-000001-hos-a'

Patient_Key,SRC_PatientID,FirstName,LastName,MiddleName,SSN,PhoneNumber,Gender,DOB,Address,SRC_ModifiedDate,datasource,is_quarantined
HOSP1-000001-hos-a,HOSP1-000001,Rick,Russo,U,188-23-9828,+1-630-829-7585x0769,Female,1937-06-04,"2701 Pablo Kisel Blvd, Brownsville, TX 78526",2020-05-25,hos-a,false


In [0]:
%sql
-- truncate table silver.patients

In [0]:
%sql
-- select * from silver.patients
-- where Patient_Key = 'HOSP1-000001-hos-a';

Patient_Key,SRC_PatientID,FirstName,LastName,MiddleName,SSN,PhoneNumber,Gender,DOB,Address,SRC_ModifiedDate,datasource,is_quarantined,inserted_date,modified_date,is_current
HOSP1-000001-hos-a,HOSP1-000001,Rick,Russo,U,188-23-9828,+1-630-829-7585x0769,Female,1937-06-04,"Unit 0915 Box 7064, DPO AA 82777",2020-05-25T00:00:00.000Z,hos-a,false,2025-12-30T23:48:53.806Z,2025-12-31T00:19:23.826Z,false
HOSP1-000001-hos-a,HOSP1-000001,Rick,Russo,U,188-23-9828,+1-630-829-7585x0769,Female,1937-06-04,"2701 Pablo Kisel Blvd, Brownsville, TX 78526",2020-05-25T00:00:00.000Z,hos-a,false,2025-12-31T00:20:12.574Z,2025-12-31T00:20:12.574Z,true


In [0]:
%sql
MERGE INTO silver.patients AS target
USING quality_checks AS source
ON target.Patient_Key = source.Patient_Key
AND target.is_current = true
-- Step 2: Insert new and updated records into the Delta table, marking them as current
WHEN NOT MATCHED
THEN INSERT (
    Patient_Key,
    SRC_PatientID,
    FirstName,
    LastName,
    MiddleName,
    SSN,
    PhoneNumber,
    Gender,
    DOB,
    Address,
    SRC_ModifiedDate,
    datasource,
    is_quarantined,
    inserted_date,
    modified_date,
    is_current
)
VALUES (
    source.Patient_Key,
    source.SRC_PatientID,
    source.FirstName,
    source.LastName,
    source.MiddleName,
    source.SSN,
    source.PhoneNumber,
    source.Gender,
    source.DOB,
    source.Address,
    source.SRC_ModifiedDate,
    source.datasource,
    source.is_quarantined,
    current_timestamp(), -- Set inserted_date to current timestamp
    current_timestamp(), -- Set modified_date to current timestamp
    true -- Mark as current
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
10000,0,0,10000


In [0]:
df_sil_patients = spark.read.format("delta").table("silver.patients")

# Define your ADLS Gen2 path
adls_path = "abfss://silver@ttadlsjcrh.dfs.core.windows.net/"

# Write the DataFrame as a Delta table
df_sil_patients.write.format("delta").mode("overwrite").save(adls_path + "/patients")